# Sri Lanka Accident Count Forecast

A single LightGBM **regression** model: given a date, it predicts how many accidents each district will have over the **next 7 days** from that date.

## 1. Imports


In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


## 2. Load Data


In [7]:
df = pd.read_csv("accidents_clean_en.csv")
df = df.drop_duplicates().dropna().reset_index(drop=True)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (4313, 19)
Columns: ['id', 'date', 'time', 'year', 'month', 'day', 'hour', 'day_of_week', 'district', 'city', 'latitude', 'longitude', 'deaths_count', 'weather_code', 'temp_max', 'temp_min', 'temp_mean', 'windgusts_max', 'humidity_mean']


## 3. Feature Engineering — One Row per (Date, District)


In [8]:
df['date'] = pd.to_datetime(df['date'])
df['district'] = df['district'].astype(str)

wcols = ['weather_code', 'temp_max', 'temp_min', 'temp_mean', 'windgusts_max', 'humidity_mean']

days = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
districts = sorted(df['district'].unique())
daily = pd.DataFrame([(d, dist) for d in days for dist in districts],
                     columns=['date', 'district'])

# Accidents and deaths per (date, district)
counts = df.groupby(['date', 'district']).size().reset_index(name='n')
deaths = df.groupby(['date', 'district'])['deaths_count'].sum().reset_index(name='deaths')
daily = daily.merge(counts, on=['date', 'district'], how='left')
daily = daily.merge(deaths, on=['date', 'district'], how='left')
daily['n'] = daily['n'].fillna(0)
daily['deaths'] = daily['deaths'].fillna(0)

# Weather = district-month climatological averages only (no leakage, any date)
clim = df.assign(m=df['date'].dt.month).groupby(['district', 'm'])[wcols].mean()
daily['m'] = daily['date'].dt.month
daily = daily.merge(clim.reset_index(), on=['district', 'm'], how='left')
dclim = df.groupby('district')[wcols].mean()
for c in wcols:
    daily[c] = daily[c].fillna(daily['district'].map(dclim[c]))
daily = daily.drop(columns=['m'])

# Calendar features
daily['year_d'] = daily['date'].dt.year
daily['month_d'] = daily['date'].dt.month
daily['day_d'] = daily['date'].dt.day

# Static district size (baseline demand)
dtot = df.groupby('district').size().astype(float)
daily['district_total'] = daily['district'].map(dtot)

# History features (strictly before the date)
daily = daily.sort_values(['district', 'date']).reset_index(drop=True)
grp = daily.groupby('district')['n']
dgrp = daily.groupby('district')['deaths']
daily['load7_prev'] = grp.transform(lambda s: s.rolling(7).sum().shift(1))
daily['deaths7_prev'] = dgrp.transform(lambda s: s.rolling(7).sum().shift(1))
daily['lag7'] = daily['load7_prev'].shift(7)
daily['lag10'] = daily['load7_prev'].shift(10)
daily['lag14'] = daily['load7_prev'].shift(14)
daily['lag21'] = daily['load7_prev'].shift(21)
daily['lag28'] = daily['load7_prev'].shift(28)
daily['lag30'] = grp.transform(lambda s: s.rolling(30).sum().shift(1))
daily['l1'] = grp.transform(lambda s: s.shift(1))
daily['deaths_lag7'] = daily['deaths7_prev'].shift(7)
daily['load364'] = daily['load7_prev'].shift(364)
daily['roll90'] = grp.transform(lambda s: s.rolling(90).sum().shift(1))
daily['roll180'] = grp.transform(lambda s: s.rolling(180).sum().shift(1))
daily['roll365'] = grp.transform(lambda s: s.rolling(365).sum().shift(1))

# Typical level for this district on this weekday / month
daily['weekday'] = daily['date'].dt.weekday
daily['dow_avg'] = daily.groupby(['district', 'weekday'])['n'].transform(
    lambda s: s.rolling(4, min_periods=1).mean().shift(1))
daily['month_avg'] = daily.groupby(['district', 'month_d'])['n'].transform(
    lambda s: s.expanding(1).mean().shift(1))

# Seasonality (day-of-year cycles)
doy = daily['date'].dt.dayofyear
daily['sindoy'] = np.sin(2 * np.pi * doy / 365.25)
daily['cosdoy'] = np.cos(2 * np.pi * doy / 365.25)

le_day = LabelEncoder()
le_day.fit(df['day_of_week'].astype(str))
daily['dow_enc'] = le_day.transform(daily['date'].dt.day_name())
le_district = LabelEncoder()
daily['district_enc'] = le_district.fit_transform(daily['district'])

# Target: accidents in the strictly NEXT 7 days [d+1, d+7]
daily['next7'] = grp.transform(lambda s: s.rolling(7).sum().shift(-7))

feats = ['year_d', 'month_d', 'day_d', 'dow_enc', 'district_enc', 'district_total',
         'lag7', 'lag10', 'lag14', 'lag21', 'lag28', 'lag30', 'l1',
         'deaths7_prev', 'deaths_lag7', 'load364', 'roll90', 'roll180', 'roll365',
         'dow_avg', 'month_avg', 'sindoy', 'cosdoy'] + wcols
print(f"Features ({len(feats)}): {feats}")


Features (29): ['year_d', 'month_d', 'day_d', 'dow_enc', 'district_enc', 'district_total', 'lag7', 'lag10', 'lag14', 'lag21', 'lag28', 'lag30', 'l1', 'deaths7_prev', 'deaths_lag7', 'load364', 'roll90', 'roll180', 'roll365', 'dow_avg', 'month_avg', 'sindoy', 'cosdoy', 'weather_code', 'temp_max', 'temp_min', 'temp_mean', 'windgusts_max', 'humidity_mean']


## 4. Train — One Regression Model


In [9]:
# Drop early rows without full history + last 7 days without a full future window
train_df = daily[daily['date'] >= daily['date'].min() + pd.Timedelta(days=400)]
train_df = train_df.dropna(subset=feats + ['next7'])

X = train_df[feats]
y = train_df['next7']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = lgb.LGBMRegressor(
    n_estimators=2500,
    learning_rate=0.01,
    max_depth=10,
    num_leaves=255,
    min_child_samples=5,
    reg_lambda=0.0,
    random_state=42,
    verbose=-1
)
model.fit(X_train, y_train)

pred = np.maximum(model.predict(X_test), 0)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Mean accidents/7days/district: {y.mean():.3f}")
print(f"MAE:  {mean_absolute_error(y_test, pred):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, pred):.3f}")
print(f"R2:   {r2_score(y_test, pred):.3f}")


Train: (52660, 29)  Test: (13165, 29)
Mean accidents/7days/district: 0.385
MAE:  0.230
RMSE: 0.376
R2:   0.762


## 5. Predict — Accidents per District for a Given Date

Full causal: only history **before** the date is used. Weather for the date uses the district-month climatological average. Returns `{'Colombo': 1, 'Gampaha': 0, ...}`.


In [ ]:
def predict_accident_count(date_str):
    """Accidents per district for the next 7 days from the given date."""
    d = pd.to_datetime(date_str)
    hist = daily[daily['date'] < d]
    clim = df.assign(m=df['date'].dt.month).groupby(['district', 'm'])[wcols].mean()
    dclim_series = df.groupby('district')[wcols].mean()
    wd = d.weekday()
    doy = d.dayofyear
    rows = []

    for dist in districts:
        h = hist[hist['district'] == dist]['n'].reset_index(drop=True)
        dh = hist[hist['district'] == dist]['deaths'].reset_index(drop=True)
        hw = hist[(hist['district'] == dist) & (hist['date'].dt.weekday == wd)]['n'].reset_index(drop=True)
        hm = hist[(hist['district'] == dist) & (hist['date'].dt.month == d.month)]['n'].reset_index(drop=True)

        if len(h) == 0:
            l1 = lag7 = lag10 = lag14 = lag21 = lag28 = lag30 = 0.0
            roll90 = roll180 = roll365 = load364 = deaths7 = deaths_lag7 = 0.0
        else:
            l1 = float(h.iloc[-1])                       # yesterday (d-1)
            lag7 = float(h.iloc[-14:-7].sum())           # [d-14, d-8]
            lag10 = float(h.iloc[-17:-10].sum())         # [d-17, d-11]
            lag14 = float(h.iloc[-21:-14].sum())         # [d-21, d-15]
            lag21 = float(h.iloc[-28:-21].sum())         # [d-28, d-22]
            lag28 = float(h.iloc[-35:-28].sum())         # [d-35, d-29]
            lag30 = float(h.tail(30).sum())              # [d-30, d-1]
            roll90 = float(h.tail(90).sum())             # [d-90, d-1]
            roll180 = float(h.tail(180).sum())           # [d-180, d-1]
            roll365 = float(h.tail(365).sum())           # [d-365, d-1]
            load364 = float(h.iloc[-371:-364].sum())     # [d-371, d-365]
            deaths7 = float(dh.tail(7).sum())            # deaths [d-7, d-1]
            deaths_lag7 = float(dh.iloc[-14:-7].sum())   # deaths [d-14, d-8]

        row = {'year_d': d.year, 'month_d': d.month, 'day_d': d.day,
               'dow_enc': le_day.transform([d.day_name()])[0],
               'district_enc': le_district.transform([dist])[0],
               'district_total': dtot[dist],
               'lag7': lag7, 'lag10': lag10, 'lag14': lag14, 'lag21': lag21,
               'lag28': lag28, 'lag30': lag30, 'l1': l1,
               'deaths7_prev': deaths7, 'deaths_lag7': deaths_lag7,
               'load364': load364, 'roll90': roll90, 'roll180': roll180,
               'roll365': roll365,
               'dow_avg': float(hw.tail(4).mean()) if len(hw) > 0 else 0.0,
               'month_avg': float(hm.mean()) if len(hm) > 0 else 0.0,
               'sindoy': np.sin(2 * np.pi * doy / 365.25),
               'cosdoy': np.cos(2 * np.pi * doy / 365.25)}
        w = clim.loc[(dist, d.month)] if (dist, d.month) in clim.index else dclim_series.loc[dist]
        for c in wcols:
            row[c] = float(w[c])
        rows.append(row)

    X_pred = pd.DataFrame(rows)
    preds = np.maximum(model.predict(X_pred[feats]), 0)
    preds = np.ceil(preds).astype(int)
    return dict(zip(districts, preds))


# Example: accidents per district from the given date (next 7 days)
result = predict_accident_count('2026-09-14')
print(result)


{'Ampara': 1, 'Anuradhapura': 1, 'Badulla': 1, 'Batticaloa': 1, 'Colombo': 1, 'Galle': 1, 'Gampaha': 1, 'Hambantota': 1, 'Jaffna': 1, 'Kalutara': 2, 'Kandy': 1, 'Kegalle': 1, 'Kilinochchi': 1, 'Kurunegala': 2, 'Mannar': 1, 'Matale': 1, 'Matara': 2, 'Monaragala': 1, 'Mullaitivu': 1, 'Nuwara Eliya': 1, 'Polonnaruwa': 1, 'Puttalam': 1, 'Ratnapura': 1, 'Trincomalee': 1, 'Vavuniya': 2}


In [17]:
import pickle
with open("accident_model.pkl", "wb") as f:
    pickle.dump(model, f)